# InsPLAD — Gold v2 Vision Expert Selection

이 notebook은 `gold_v2_final` 246장을 직접 튜닝에 쓰지 않고, 기존 50장 development smoke에서 Vision Expert와 evidence 형식을 선택합니다.

비교:
- Vision Expert: PatchCore / Dinomaly / AnomalyCLIP / ReConPatch-lite
- Evidence: score / bbox / heatmap
- LVLM: Qwen3.5-4B / InternVL3.5-8B

선정 기준:
1. 두 LVLM의 mean anomaly F1
2. 동률이면 mean balanced accuracy
3. 그래도 동률이면 invalid output rate가 낮은 조합

선정된 winner만 이후 `gold_v2_final` 246장 전체에 확장합니다.

전체 artifact는 Drive에 저장하고 핵심 결과는 GitHub `results/latest/gold_v2_vision_expert_selection/`에 자동 업로드합니다.


In [ ]:
# 0) 설정
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"

PREVIOUS_RUN_FOLDER = "vision_expert_smoke_v1"
RUN_FOLDER = "gold_v2_vision_expert_selection"

SMOKE_SIZE = 50
SEED = 42

MODELS_TO_RUN = ["qwen3_5_4b", "internvl3_5_8b"]
EXPERTS_TO_RUN = ["patchcore", "dinomaly", "anomalyclip", "reconpatch_lite"]
CONDITIONS = ["score", "bbox", "heatmap"]

RECONPATCH_TRAIN_STEPS = 120
RECONPATCH_MAX_TRAIN_PATCHES = 6000
RECONPATCH_MEMORY_RATIO = 0.05
RECONPATCH_BATCH_SIZE = 192
CHECKPOINT_EVERY = 1

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
EXPECTED_UNSUP_ZIP_SIZE = 1_170_967_719


In [ ]:
# 1) Drive + repo clone + 환경 준비
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
RESULT_BASE = DRIVE_ROOT / "lvlm_insplad_results"
PREV_ROOT = RESULT_BASE / PREVIOUS_RUN_FOLDER
RUN_ROOT = RESULT_BASE / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:10]) + (" ..." if len(cmd) > 10 else ""))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 없습니다.")
os.environ["GITHUB_TOKEN"] = github_token

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT": "0",
})
run([
    "git", "clone", "--depth", "1", "--branch", REPO_REF,
    f"https://github.com/{REPO}.git", REPO_DIR
], env=git_env)
del auth, git_env

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.14.0",
    "accelerate==1.14.0",
    "sentencepiece==0.2.2",
    "pandas>=2.2",
    "numpy>=1.26",
    "pillow>=10",
    "tqdm>=4.66",
])
run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev]"])

repo_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print("repo:", repo_commit)
print("previous smoke:", PREV_ROOT)
print("run:", RUN_ROOT)


In [ ]:
# 2) 기존 50장 development smoke 결과 + source ZIP 확인
import pandas as pd
import torch

required = {
    "qa": PREV_ROOT / "data/aligned_qa" / f"one_shot_plus_smoke_{SMOKE_SIZE}.jsonl",
    "workspace": PREV_ROOT / "workspace" / "workspace_manifest.csv",
    "workspace_config": PREV_ROOT / "workspace" / "workspace_config.json",
    "base_roi_manifest": PREV_ROOT / "vision_experts" / "roi_manifest.csv",
}
missing = [f"{name}: {path}" for name, path in required.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "기존 vision expert smoke 결과가 없습니다. 먼저 "
        "`notebooks/lvlm_vision_expert_smoke_resumable_colab.ipynb`를 한 번 완료하세요.\n"
        + "\n".join(missing)
    )

SMOKE_QA = required["qa"]
WORKSPACE = PREV_ROOT / "workspace"
BASE_ROI_MANIFEST = required["base_roi_manifest"]
BASE_ARTIFACT_ROOT = PREV_ROOT

def find_unique_file(filename):
    matches = [p for p in DRIVE_ROOT.rglob(filename) if p.is_file()]
    unique = []
    seen = set()
    for p in matches:
        key = str(p.resolve())
        if key not in seen:
            unique.append(p)
            seen.add(key)
    if not unique:
        raise FileNotFoundError(filename)
    if len(unique) > 1:
        raise RuntimeError(
            f"{filename}이 여러 개 있습니다. 중복을 정리하세요: "
            + ", ".join(str(x) for x in unique[:10])
        )
    return unique[0]

UNSUP_ZIP = find_unique_file(UNSUP_ZIP_NAME)
if UNSUP_ZIP.stat().st_size != EXPECTED_UNSUP_ZIP_SIZE:
    raise RuntimeError(f"unsupervised ZIP size mismatch: {UNSUP_ZIP.stat().st_size}")

base_roi = pd.read_csv(BASE_ROI_MANIFEST)
print(base_roi.groupby("expert").size())
for expert in ["patchcore", "dinomaly", "anomalyclip"]:
    count = int((base_roi["expert"] == expert).sum())
    if count != SMOKE_SIZE:
        raise RuntimeError(f"{expert}: expected {SMOKE_SIZE} ROI rows, got {count}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime으로 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("smoke QA:", SMOKE_QA)


In [ ]:
# 3) ReConPatch-lite 추가 — 이미 완성되어 있으면 재사용
RECON_ROOT = RUN_ROOT / "reconpatch_lite"
RECON_MANIFEST = RECON_ROOT / "roi_manifest_reconpatch_lite.csv"

def recon_complete():
    if not RECON_MANIFEST.is_file():
        return False
    df = pd.read_csv(RECON_MANIFEST)
    return len(df) == SMOKE_SIZE and set(df["expert"]) == {"reconpatch_lite"}

if recon_complete():
    print("재사용:", RECON_MANIFEST)
else:
    run([
        sys.executable, REPO_DIR / "scripts/run_reconpatch_lite_smoke.py",
        "--workspace-root", WORKSPACE,
        "--out-dir", RECON_ROOT,
        "--train-steps", str(RECONPATCH_TRAIN_STEPS),
        "--max-train-patches", str(RECONPATCH_MAX_TRAIN_PATCHES),
        "--memory-ratio", str(RECONPATCH_MEMORY_RATIO),
        "--batch-size", str(RECONPATCH_BATCH_SIZE),
        "--seed", str(SEED),
    ])
    if not recon_complete():
        raise RuntimeError("ReConPatch-lite 50장 결과가 완성되지 않았습니다.")

print("ReConPatch-lite:", RECON_MANIFEST)


In [ ]:
# 4) 4개 expert의 score / bbox / heatmap evidence 생성
EVIDENCE_ROOT = RUN_ROOT / "evidence"
EVIDENCE_MANIFEST = EVIDENCE_ROOT / "evidence_manifest.csv"

run([
    sys.executable, REPO_DIR / "scripts/build_evidence_ablation.py",
    "--qa", SMOKE_QA,
    "--workspace-root", WORKSPACE,
    "--base-roi-manifest", BASE_ROI_MANIFEST,
    "--base-artifact-root", BASE_ARTIFACT_ROOT,
    "--reconpatch-manifest", RECON_MANIFEST,
    "--reconpatch-artifact-root", RECON_ROOT,
    "--out-dir", EVIDENCE_ROOT,
])

evidence = pd.read_csv(EVIDENCE_MANIFEST)
expected_rows = SMOKE_SIZE * len(EXPERTS_TO_RUN)
if len(evidence) != expected_rows:
    raise RuntimeError(f"evidence rows mismatch: {len(evidence)} != {expected_rows}")
if set(evidence["expert"]) != set(EXPERTS_TO_RUN):
    raise RuntimeError(f"expert coverage mismatch: {sorted(evidence['expert'].unique())}")

display(evidence.groupby("expert").size().rename("rows"))
print("evidence:", EVIDENCE_MANIFEST)


In [ ]:
# 5) Qwen + InternVL에서 4 expert × 3 evidence 비교
EVAL_DIR = RUN_ROOT / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

run([
    sys.executable, REPO_DIR / "scripts/run_hf_evidence_ablation.py",
    "--qa", SMOKE_QA,
    "--evidence-manifest", EVIDENCE_MANIFEST,
    "--artifact-root", EVIDENCE_ROOT,
    "--workspace-root", WORKSPACE,
    "--unsup-zip", UNSUP_ZIP,
    "--out-dir", EVAL_DIR,
    "--models", *MODELS_TO_RUN,
    "--experts", *EXPERTS_TO_RUN,
    "--conditions", *CONDITIONS,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
])


In [ ]:
# 6) Screening 요약 + winner 고정
leaderboard = pd.read_csv(EVAL_DIR / "leaderboard.csv")
screen = leaderboard[
    leaderboard["model_key"].isin(MODELS_TO_RUN)
    & leaderboard["expert"].isin(EXPERTS_TO_RUN)
    & leaderboard["condition"].isin(CONDITIONS)
].copy()

expected = len(MODELS_TO_RUN) * len(EXPERTS_TO_RUN) * len(CONDITIONS)
if len(screen) != expected:
    raise RuntimeError(f"leaderboard coverage mismatch: {len(screen)} != {expected}")

summary = (
    screen.groupby(["expert", "condition"], as_index=False)
    .agg(
        models=("model_key", "nunique"),
        mean_f1=("f1", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_invalid_output_rate=("invalid_output_rate", "mean"),
    )
)
summary = summary.sort_values(
    ["mean_f1", "mean_balanced_accuracy", "mean_invalid_output_rate"],
    ascending=[False, False, True],
).reset_index(drop=True)

summary.to_csv(EVAL_DIR / "screening_summary.csv", index=False)

winner_row = summary.iloc[0].to_dict()
winner = {
    "winner_expert": winner_row["expert"],
    "winner_condition": winner_row["condition"],
    "mean_f1": float(winner_row["mean_f1"]),
    "mean_balanced_accuracy": float(winner_row["mean_balanced_accuracy"]),
    "mean_accuracy": float(winner_row["mean_accuracy"]),
    "mean_invalid_output_rate": float(winner_row["mean_invalid_output_rate"]),
    "selection_rule": (
        "highest mean anomaly F1 across Qwen3.5-4B and InternVL3.5-8B; "
        "balanced accuracy then invalid-output rate break ties"
    ),
    "selection_scope": "50-image development smoke only; not final gold_v2 paper score",
    "final_gold_v2_used_for_selection": False,
}
(EVAL_DIR / "winner.json").write_text(
    json.dumps(winner, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

metadata = {
    "protocol": "gold_v2_vision_expert_method_selection",
    "repo_commit": repo_commit,
    "selection_dataset": "existing 50-image development smoke",
    "selection_size": SMOKE_SIZE,
    "final_gold_v2_used_for_selection": False,
    "models": MODELS_TO_RUN,
    "experts": EXPERTS_TO_RUN,
    "conditions": CONDITIONS,
    "winner": winner,
    "next_step": (
        "freeze winner expert/evidence and evaluate that one configuration on "
        "gold_v2_final 246 images, then combine with Normal Reference"
    ),
    "reconpatch_warning": (
        "ReConPatch-lite is screening-only. If it wins, revalidate with a "
        "publication-grade implementation before paper claims."
    ),
}
(EVAL_DIR / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("Winner")
print(json.dumps(winner, ensure_ascii=False, indent=2))
display(summary)


In [ ]:
# 7) 핵심 결과 GitHub 자동 publish — ChatGPT가 바로 읽을 수 있음
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", EVAL_DIR,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_vision_expert_selection",
    "--files",
    "leaderboard.csv",
    "anomaly_by_asset.csv",
    "run_status.json",
    "screening_summary.csv",
    "winner.json",
    "run_metadata.json",
], env=os.environ.copy())

print("완료: results/latest/gold_v2_vision_expert_selection/")


## 다음 단계

끝나면 ChatGPT에 `vision expert 결과 나옴`이라고 하면 됩니다.

그다음:
1. 선택된 expert + evidence 조합을 고정
2. `gold_v2_final` 246장 전체에서 최종 평가
3. winner + Normal Reference 결합
4. Agent 구조 비교
